# KT AIVLE 미니프로젝트 1차
팀 모델링 코드를 로컬 환경에서 실행할 수 있도록 정리한 노트북입니다. 데이터 준비 및 실행 방법은 저장소 README를 참고하세요.

#**스마트폰 센서 데이터 기반 모션 분류**
# 단계1 : 탐색적 데이터 분석

## 0.미션

### (1) 미션1
수많은 feature들을 모두 살펴보는 것은 과도한 시간과 노력이 필요합니다. 그래서 여러분은 **선택과 집중**을 해야 합니다.
* 1) **기본 모델을 생성**한 후 **변수 중요도**를 구합니다.   
    * random forest 알고리즘 사용을 권장합니다.
* 2) **중요한 feature와 중요하지 않은 feature 상위 N개를 선정**하고, 이들을 대상으로 EDA 수행.


### (2) 미션2
다음의 case에 맞게 feature 및 feature 그룹 중요도를 기반으로 탐색적 데이터 분석을 수행하시오.
* Target을 **정적/동적 행동**으로 구분
    * 6개의 행동은 2개의 그룹(정적행동, 동적행동)으로 나뉩니다.  
    * 어떤 feature(혹은 feature 그룹)이 2개 class 그룹(정적행동, 동적행동)를 구분하는데 중요한지를 찾아보고 탐색해봅시다.



### (3) 추가 과제
다음의 case에 맞게 feature 및 feature 그룹 중요도를 기반으로 탐색적 데이터 분석을 수행하시오.
* 1) Target을 **개별 행동 여부**로 구분
    * 6가지의 행동을 분류하는 분석도 중요하지만, 개별 행동에만 특별히 영향을 받는 feature들도 있습니다.
    * 예를 들어, 계단을 오르는 행동(Walking_upstairs)과 관련이 큰 feature가 있을 것입니다. [계단을 오르는 행동]인지 아닌지로 구분하는 targe을 추가하여 EDA를 수행해 봅시다.


## 1.환경설정

* 세부 요구사항
    - 경로 설정 : 구글콜랩
        * 구글 드라이브 바로 밑에 project1 폴더를 만들고,
        * 데이터 파일을 복사해 넣습니다.
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
        * 필요하다고 판단되는 라이브러리를 추가하세요.

### (1) 경로 설정

* 구글 드라이브 연결

In [ ]:
# 로컬 실행용 사본: Google Drive 마운트 생략


In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'requirements.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
path = str(ROOT / 'data' / 'har') + '/'


In [ ]:
import os
os.chdir(path)


### (2) 라이브러리 설치 및 불러오기

* 라이브러리 로딩

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder


* 제공 함수 생성
    * 변수 중요도를 시각화할 수 있는 함수를 제공합니다.
    * 입력 :
        * importance : 트리모델의 변수 중요도(예: model.feature_importances_)
        * names : 변수 이름 목록(예 : x_train.columns)
        * result_only  : 변수 중요도 순으로 데이터프레임만 return할지, 그래프도 포함할지 결정. False이면 결과 데이터프레임 + 그래프
        * topn : 중요도 상위 n개만 표시. all 이면 전체.
    * 출력 :
        * 중요도 그래프 : 중요도 내림차순으로 정렬
        * 중요도 데이터프레임 : 중요도 내림차순으로 정렬

In [ ]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (3) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
    * data01_test.csv : 테스트용
    * feature.csv : feature 이름을 계층구조로 정리한 데이터

* 세부 요구사항
    * 칼럼 삭제 : data01_train.csv와 data01_test.csv 에서 'subject' 칼럼은 불필요하므로 삭제합니다.

In [ ]:
# 주어진 데이터 셋을 불러오세요.
train = pd.read_csv('data01_train.csv')
test = pd.read_csv('data01_test.csv')
features = pd.read_csv('features.csv')

In [ ]:
#불필요한 칼럼을 삭제하세요.
for i in (train, test):
    i.drop('subject', axis=1, inplace=True)

## 2.데이터 분석 : 기본정보 살펴보기

### (1) 기본정보 확인하기


* **세부 요구사항**
    - 불러온 데이터의 형태, 기초통계량, 정보 등을 확인합니다.

In [ ]:
# 전체 데이터의 행, 열 개수를 확인
print(train.shape)
print(test.shape)

In [ ]:
# 전체 데이터의 상위 5개 행 확인
train.head()

In [ ]:
# 전체 데이터의 수치형 변수 분포 확인
train.describe()

In [ ]:
#전체 데이터의 모든 변수 확인
train.info()

In [ ]:
#(추가) 다양하게 기본 정보들을 확인하세요!
features.value_counts()

### (2) 목표 변수 확인하기

* **세부 요구사항**
    - target인 'Activity'에 대해 단변량 분석을 수행합니다.
    - 범주의 종류, 범주별 빈도수 및 비율 등

In [ ]:
# Activity 에 대해서 단변량 분석을 시행합니다. (범주별 빈도수, 종류, 비율, 시각화도 해보기!)
target = train['Activity']

In [ ]:
target.value_counts()

In [ ]:
plt.pie(target.value_counts(), labels = target.value_counts().index)
plt.show()

In [ ]:
data01_train_Activity = train['Activity'].value_counts()
data01_test_Activity = test['Activity'].value_counts()

plt.bar(x=data01_train_Activity.index.astype(str), height=data01_train_Activity.values)
plt.bar(x=data01_test_Activity.index.astype(str), height=data01_test_Activity.values)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### (3) 센서 그룹 데이터: features

* **세부 요구사항**
    - feature 이름에 대한 계층 구조별로 변수의 수를 카운트 해 봅시다.
        - sensor 별 변수의 개수
        - agg(regation) 별 변수의 개수
        - axis 별 변수의 개수

In [ ]:
#센서 그룹 데이터의 'sensor'변수의 값 확인
features['sensor'].value_counts()

In [ ]:
#센서 그룹 데이터의 'agg'변수의 값 확인
features['agg'].value_counts()

In [ ]:
#센서 그룹 데이터의 'axis'변수의 값 확인
features['axis'].value_counts()

## 3.데이터 분석 : 미션1

수많은 feature들을 모두 살펴보는 것은 과도한 시간과 노력이 필요합니다. 그래서 여러분은 **선택과 집중**을 해야 합니다.
* 1) **기본 모델을 생성**한 후 **변수 중요도**를 구합니다.   
    * random forest 알고리즘 사용을 권장합니다.
* 2) **중요한 feature와 중요하지 않은 feature 상위 N개를 선정**하고, 이들을 대상으로 EDA 수행.
* 3) **각 feature 그룹별 중요도**도 파악해보며 EDA를 수행  
    * Feature 이름에는 계층구조를 담고 있습니다. 그렇다 보니 feature들을 적절하게 그룹으로 묶을 수 있습니다.
    * 참고로, feature 그룹의 중요도는 개별 feature 중요도의 합으로 계산할 수 있습니다.

### (1) 기본모델링 + 변수중요도 추출

* **세부 요구사항**
    - 모델링을 위한 기본 전처리를 수행합니다.
        - 주어진 데이터에서는 x, y로 나누고 train과 validation 분할 수행
    - 트리기반 알고리즘을 이용하여 모델링을 합니다.
        - 랜덤포레스트 권장 : 랜덤포레스트는 튜닝 없이 하이퍼파리미터의 기본값로도 적절한 성능의 모델을 생성할 수 있습니다.
    - 변수 중요도 추출
        - 사용자정의함수로 생성한 plot_feature_importance 을 이용
        - 변수중요도 파악및 결과 저장

#### 1) 데이터 전처리

In [ ]:
# 데이터 분할을 위한 전처리
train['is_dynamic'] = train.loc[:, 'Activity'].map(lambda a: 0 if a in ['LAYING', 'STANDING', 'SITTING'] else 1)
test['is_dynamic'] = test.loc[:, 'Activity'].map(lambda a: 0 if a in ['LAYING', 'STANDING', 'SITTING'] else 1)

le = LabelEncoder()
train['y2'] = le.fit_transform(train['Activity'])
test['y2'] = le.transform(test['Activity'])

X = train.drop(['Activity', 'is_dynamic', 'y2'], axis=1)
y = train.loc[:, 'Activity']
is_dynamic = train.loc[:, 'is_dynamic']
y2 = train.loc[:, 'y2']

X_train, X_val, y_train, y_val = train_test_split(X, y2, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

#### 2) 모델링

In [ ]:
#생성
baseline = RandomForestClassifier(random_state=42)
#학습
baseline.fit(X_train, y_train)
#평가
y_pred = baseline.predict(X_val)
print(confusion_matrix(y_val, y_pred))

#### 3) 변수 중요도 추출

In [ ]:
# 변수 중요도 추출(시각화로 나타내기)

In [ ]:
# 변수 중요도 추출(시각화로 나타내기)
y2_fi = plot_feature_importance(baseline.feature_importances_, baseline.feature_names_in_, topn='all')

### (2) 중요도 기반 feature 분석

* **세부 요구사항**
    - 상위 n, 하위 m 개의 변수를 선정
    - 선정된 변수(feature)와 target 간의 관계를 kdeplot으로 그리기

#### 1) 상위 5, 하위 5개 변수 선정

In [ ]:
# 중요도 상위 top 5
top5 = plot_feature_importance(baseline.feature_importances_, baseline.feature_names_in_, result_only=True, topn=5)
top5

In [ ]:
# 중요도 하위 top 5
low5 = plot_feature_importance(baseline.feature_importances_, baseline.feature_names_in_, result_only=True, topn='all')[-5:]
low5

#### 2) 상위 5개 변수에 대한 분석
        - 위에서 나온 상위 5개의 변수를 선정해서 target과의 관계를 kdeplot으로 그려보세요!

In [ ]:
#1위 :	tGravityAcc-mean()-X
top_feature = top5.iloc[0]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()


In [ ]:
#2위: angle(X,gravityMean)
top_feature = top5.iloc[1]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#3위: tGravityAcc-max()-X
top_feature = top5.iloc[2]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#4위: tGravityAcc-mean()-Y
top_feature = top5.iloc[3]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#5위: tGravityAcc-min()-X
top_feature = top5.iloc[4]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

#### 3) 하위 5개 변수에 대한 분석
        - 위에서 하위 5개의 변수를 선정해서 target과의 관계를 kdeplot으로 그려보세요!

In [ ]:
#1위 : tBodyAccJerkMag-arCoeff()3
top_feature = low5.iloc[-1]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#2위: tGravityAccMag-max()
top_feature = low5.iloc[-2]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#3위: tGravityAccMag-min()
top_feature = low5.iloc[-3]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#4위: tGravityAccMag-energy()
top_feature = low5.iloc[-4]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

In [ ]:
#5위: tGravityAccMag-iqr()
top_feature = low5.iloc[-5]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='y2')
plt.show()

### (3) feature 그룹별 분석

* **세부 요구사항**
    - 제공된 features.csv 에는 feature에 대한 계층구조를 정리해 놓았습니다.
    - features와 변수중요도 결과를 merge 하시오.
    - sensor 별 중요도 합계를 구하고 상위 변수 그룹별 비교 분석을 수행 하시오.
    - sensor + agg 별 중요도 합계를 구하고 상위 변수 그룹별 비교 분석을 수행 하시오.


#### 1) 결과 merge

In [ ]:
merged_features = pd.merge(features, y2_fi, on='feature_name')

#### 2) sensor 별 중요도

In [ ]:
# sensor 별 중요도 그래프 그리기
sorted_df = merged_features.groupby('sensor', as_index=False)['feature_importance'].mean().sort_values('feature_importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=sorted_df, x='feature_importance', y='sensor')
plt.show()

In [ ]:
# agg 별 중요도 그래프 그리기
sorted_df = merged_features.groupby('agg', as_index=False)['feature_importance'].mean().sort_values('feature_importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=sorted_df, x='feature_importance', y='agg')
plt.show()

#### 3) sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회하여 그래프 그리기
merged_features['sensor+agg'] = merged_features['sensor'] + '+' + merged_features['agg']
sorted_df = merged_features.groupby('sensor+agg', as_index=False)['feature_importance'].mean().sort_values('feature_importance', ascending=False).head(20)
plt.figure(figsize=(10,6))
sns.barplot(data=sorted_df, x='feature_importance', y='sensor+agg')
plt.show()

In [ ]:
# Sensor 별, agg로 나눠서 분석하기
sensors = merged_features['sensor'].unique()
n = len(sensors)

cols = 3
rows = (n + cols - 1) // cols
plt.figure(figsize=(15, 4 * rows))

for i, sensor in enumerate(sensors):
    plt.subplot(rows, cols, i + 1)
    subset = merged_features[merged_features['sensor'] == sensor]
    sns.barplot(
        data=subset,
        x='feature_importance',
        y='agg',
        hue='agg',
        estimator='mean',
    )
    plt.title(f'{sensor} (mean importance)')
    plt.xlim(0, 0.04)
plt.tight_layout()
plt.show()


## 4.데이터 분석 : 미션2

다음의 case에 맞게 feature 및 feature 그룹 중요도를 기반으로 탐색적 데이터 분석을 수행하시오.
* Target을 **정적/동적 행동**으로 구분
    * 6개의 행동은 2개의 그룹(정적행동, 동적행동)으로 나뉩니다.  
    * 어떤 feature(혹은 feature 그룹)이 2개 class 그룹(정적행동, 동적행동)를 구분하는데 중요한지를 찾아보고 탐색해봅시다.

### (1) 정적/동적 행동으로 구분하여 분석하기

* **세부 요구사항**
    - Target인 Activity를 is_dynamic 변수로 변경하시오.
        * 값 0 : STANDING, SITTING, LAYING
        * 값 1 : WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS
    - is_dynamic을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_dynamic을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_dynamic 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_dynamic 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_dynamic 추가

In [ ]:
#is dynamic 변수를 추가하세요(값 0, 값 1 확인해서!)

#### 2) 기본모델 생성

In [ ]:
#데이터 분할 진행(train:val = 8:2 혹은 7:3 권장)
X_train, X_val, y_train, y_val = train_test_split(X, is_dynamic, test_size=0.3, random_state=42)

In [ ]:
#RandomForestClassifier로 모델링 진행
#RandomForestClassifier로 모델링 진행
#생성
baseline = RandomForestClassifier(random_state=42)
#학습
baseline.fit(X_train, y_train)
#평가
y_pred = baseline.predict(X_val)
print(confusion_matrix(y_val, y_pred))

In [ ]:
# 데이터 분할을 위한 전처리
train['is_dynamic'] = train.loc[:, 'Activity'].map(lambda a: 0 if a in ['LAYING', 'STANDING', 'SITTING'] else 1)
test['is_dynamic'] = test.loc[:, 'Activity'].map(lambda a: 0 if a in ['LAYING', 'STANDING', 'SITTING'] else 1)

le = LabelEncoder()
train['y2'] = le.fit_transform(train['Activity'])
test['y2'] = le.transform(test['Activity'])

X = train.drop(['Activity', 'is_dynamic', 'y2'], axis=1)
y = train.loc[:, 'Activity']
is_dynamic = train.loc[:, 'is_dynamic']
y2 = train.loc[:, 'y2']

X_train, X_val, y_train, y_val = train_test_split(X, y2, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, is_dynamic, test_size=0.3, random_state=42)


#### 3) 변수중요도 상위 5개 분석

In [ ]:
# is_dynamic 을 구분하는데 중요한 변수 상위 5를 분석!
top5 = plot_feature_importance(baseline.feature_importances_, baseline.feature_names_in_, result_only=True, topn=5)
top5

* 미션1에서 했던 것 처럼 상위 5개의 변수 데이터를 탐색해보기!

In [ ]:
train

In [ ]:
#1위 :	fBodyAccJerk-max()-X
top_feature = top5.iloc[0]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='is_dynamic')
plt.show()

In [ ]:
#2위:tBodyGyroJerk-iqr()-Z
top_feature = top5.iloc[1]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='is_dynamic')
plt.show()


In [ ]:
#3위: fBodyAccJerk-bandsEnergy()-1,16
top_feature = top5.iloc[2]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='is_dynamic')
plt.show()


In [ ]:
#4위: tBodyAccJerk-std()-X
top_feature = top5.iloc[3]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='is_dynamic')
plt.show()


In [ ]:
#5위: fBodyAcc-entropy()-X
top_feature = top5.iloc[4]['feature_name']
sns.kdeplot(data=train, x=top_feature, hue='is_dynamic')
plt.show()


#### 4) feature 그룹별 분석

* 결과 merge

In [ ]:
dynamic_features = plot_feature_importance(baseline.feature_importances_, baseline.feature_names_in_, result_only=True, topn='all')
dynamic_features.rename(columns={'feature_importance': 'is_dynamic_importance'}, inplace=True)
merged_features = pd.merge(merged_features, dynamic_features, on='feature_name')

In [ ]:
merged_features

* sensor 별 중요도

In [ ]:
# sensor 별 중요도 그래프 그리기
sorted_df = merged_features.groupby('sensor', as_index=False)['is_dynamic_importance'].mean().sort_values('is_dynamic_importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=sorted_df, x='is_dynamic_importance', y='sensor')
plt.show()

In [ ]:
# agg 별 중요도 그래프 그리기
sorted_df = merged_features.groupby('agg', as_index=False)['is_dynamic_importance'].mean().sort_values('is_dynamic_importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=sorted_df, x='is_dynamic_importance', y='agg')
plt.show()

In [ ]:
# 상위 20개만 조회하여 그래프 그리기
merged_features['sensor+agg'] = merged_features['sensor'] + '+' + merged_features['agg']
sorted_df = merged_features.groupby('sensor+agg', as_index=False)['is_dynamic_importance'].mean().sort_values('is_dynamic_importance', ascending=False).head(20)
plt.figure(figsize=(10,6))
sns.barplot(data=sorted_df, x='is_dynamic_importance', y='sensor+agg')
plt.show()

In [ ]:
# Sensor 별, agg로 나눠서 분석하기
sensors = merged_features['sensor'].unique()
n = len(sensors)

cols = 3
rows = (n + cols - 1) // cols
plt.figure(figsize=(15, 4 * rows))

for i, sensor in enumerate(sensors):
    plt.subplot(rows, cols, i + 1)
    subset = merged_features[merged_features['sensor'] == sensor]
    sns.barplot(
        data=subset,
        x='is_dynamic_importance',
        y='agg',
        hue='agg',
        estimator='mean',
    )
    plt.title(f'{sensor} (mean importance)')
    plt.xlim(0, 0.04)
plt.tight_layout()
plt.show()


### 5.데이터 분석 : 추가 과제

다음의 case에 맞게 feature 및 feature 그룹 중요도를 기반으로 탐색적 데이터 분석을 수행하시오.
* Target을 **개별 행동 여부**로 구분
    * 6가지의 행동을 분류하는 분석도 중요하지만, 개별 행동에만 특별히 영향을 받는 feature들도 있습니다.
    * 예를 들어, 계단을 오르는 행동(Walking_upstairs)과 관련이 큰 feature가 있을 것입니다. [계단을 오르는 행동]인지 아닌지로 구분하는 targe을 추가하여 EDA를 수행해 봅시다.

### (2) 개별 동작 분석 : is_standing


* **세부 요구사항**
    - Target인 Activity를 is_standing 변수로 변경하시오.
        * 값 1 : STANDING
        * 값 0 : 나머지
    - is_standing 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_standing 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_standing 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_standing 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_standing 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회


In [ ]:
# Sensor 별, agg로 나눠서 분석하기


### (3) 개별 동작 분석 : is_sitting


* **세부 요구사항**
    - Target인 Activity를 is_sitting 변수로 변경하시오.
        * 값 1 : SITTING
        * 값 0 : 나머지
    - is_sitting 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_sitting 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_sitting 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_sitting 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_sitting 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회


In [ ]:
# Sensor 별, agg로 나눠서 분석하기


### (4) 개별 동작 분석 : is_laying


* **세부 요구사항**
    - Target인 Activity를 is_laying 변수로 변경하시오.
        * 값 1 : LAYING
        * 값 0 : 나머지
    - is_laying 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_laying 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_laying 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_laying 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_laying 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회


In [ ]:
# Sensor 별, agg로 나눠서 분석하기


### (5) 개별 동작 분석 : is_walking


* **세부 요구사항**
    - Target인 Activity를 is_walking 변수로 변경하시오.
        * 값 1 : WALKING
        * 값 0 : 나머지
    - is_walking 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_walking 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_walking 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_walking 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_walking 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회


In [ ]:
# Sensor 별, agg로 나눠서 분석하기


### (6) 개별 동작 분석 : is_walking_up


* **세부 요구사항**
    - Target인 Activity를 is_walking_up 변수로 변경하시오.
        * 값 1 : WALKING_UPSTAIRS
        * 값 0 : 나머지
    - is_walking_up 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_walking_up 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_walking_up 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_walking_up 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_walking_up 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# Sensor 별, agg로 나눠서 분석하기


### (7) 개별 동작 분석 : is_walking_down


* **세부 요구사항**
    - Target인 Activity를 is_walking_down 변수로 변경하시오.
        * 값 1 : WALKING_DOWNSTAIRS
        * 값 0 : 나머지
    - is_walking_down 을 예측하기 위한 기본 모델을 생성하시오.
        * x, is_walking_down 을 이용하여 train과 validation으로 데이터 분할
        * 기본 모델링
    - is_walking_down 을 구분하는데 중요한 변수 상위 5를 분석하시오.
    - is_walking_down 을 구분하는데 중요한 변수 그룹을 분석하시오.
        * sensor별
        * sensor+agg 별


#### 1) is_walking_up 추가

#### 2) 기본모델 생성

#### 3) 변수중요도 상위 5개 분석

* 상위 변수 선정

* 데이터 탐색하기

In [ ]:
# 1위 :

In [ ]:
# 2위

#### 4) feature 그룹별 분석

* 결과 merge

* sensor 별 중요도

* sensor + agg 별 중요도

In [ ]:
# 상위 20개만 조회


In [ ]:
# Sensor 별, agg로 나눠서 분석하기


## 5.특성 중요도 저장하기

* **세부 요구사항**
    - 지금까지 변수 중요도를 다양한 관점에서 뽑고 분석을 해 보았습니다.
        - 관점1 : 6개 행동 구분
        - 관점2 : 동적, 정적 행동 구분
        - 관점3 : Standing 여부 구분
        - 관점4 : Sitting 여부 구분
        - 관점5 : Laying 여부 구분
        - 관점6 : Walking 여부 구분
        - 관점7 : Walking_upstairs 여부 구분
        - 관점8 : Walking_downstairs 여부 구분
    - 모든 관점의 변수 중요도를 하나로 합친 후 저장
        - merge를 이용하여 features와 합칩니다.
        - joblib 패키지의 dump를 이용하여 결과를 저장합니다.

### (1) 하나로 합치기
* 합치기 위해서는 중요도 결과에서 feature_importance 칼럼 이름을 수정해야 합니다.
* pd.merge를 이용해서 하나씩 합치는 작업을 반복 합니다.

#### 1) 이름변경

In [ ]:
# tip : rename을 써서 칼럼 이름 수정을 해야합니다!
target_groups = {
    "is_laying": ["LAYING"],
    "is_standing": ["STANDING"],
    "is_sitting": ["SITTING"],
    "is_walking": ["WALKING"],
    "is_walking_up": ["WALKING_UPSTAIRS"],
    "is_walking_down": ["WALKING_DOWNSTAIRS"]
}

exclude_cols = {"Activity", "is_dynamic", "y2"}
X_cols = [c for c in train.columns if c not in exclude_cols]
X_all = train[X_cols].copy()

# 각 클래스별로 반복
for col_name, label_list in target_groups.items():
    train[col_name] = train["Activity"].isin(label_list).astype(int)
    test[col_name] = test["Activity"].isin(label_list).astype(int)
    y_bin = train[col_name]
    X_tr, X_va, y_tr, y_va = train_test_split(X_all, y_bin, test_size=0.3, random_state=42)

    baseline = RandomForestClassifier(random_state=42)
    baseline.fit(X_tr, y_tr)
    fi_df = pd.DataFrame({"feature_name": baseline.feature_names_in_, f"{col_name}_importance": baseline.feature_importances_})
    merged_features = merged_features.merge(fi_df, on="feature_name", how="left")

In [ ]:
merged_features = merged_features.drop('sensor+agg', axis=1)

In [ ]:
train = train.drop('Activity', axis=1)
test = test.drop('Activity', axis=1)
train = train.rename(columns={"y2": 'Activity'})
test = test.rename(columns={"y2": 'Activity'})

#### 2) 합치기

In [ ]:
# merge 해서 2개를 합치는 작업하기

### (2) 저장하기
* joblib.dump를 이용하시오.
* 저장할 파일의 확장자는 보통 .pkl 입니다.

In [ ]:
joblib.dump(merged_features, 'features.pkl')

In [ ]:
# joblib.dump 이용해서 colab에 .pkl 파일에 저장하기(tip: path + '파일 이름.pkl')
joblib.dump(train, 'train.pkl')
joblib.dump(test, 'test.pkl')